# Q-factorisation on Four Rooms

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.fourrooms_discrete import FourRoomsGridWorld, FourRoomsGoalWrapper
from utils import TrajectoryReplayBufferDiscrete, evaluate_policy
from visualisations import plot_policy_rollouts, plot_q_diagnostics
from networks import DQN_QNetwork


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
class FactorisedDQN_QNetwork(nn.Module):

    def __init__(
        self,
        obs_dim: int,
        num_actions: int,
        goal_dim: int = 2,
        hidden_dim: int = 128,
        rep_dim: int = 64,
    ):
        super().__init__()
        self.num_actions = num_actions
        self.goal_dim = goal_dim
        self.rep_dim = rep_dim

        # Environment / state encoder: s -> phi_s(s) in R^rep_dim
        self.obs_encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
            nn.ReLU(),
        )

        # Action embedding: a -> e_a in R^rep_dim
        self.action_emb = nn.Embedding(num_actions, rep_dim)

        # Goal / task encoder: z -> psi(z) in R^rep_dim
        self.goal_encoder = nn.Sequential(
            nn.Linear(goal_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
        )

    def forward(self, obs: torch.Tensor, goal: torch.Tensor) -> torch.Tensor:
        B = obs.shape[0]

        phi_s = torch.tanh(self.obs_encoder(obs))
        psi_z = torch.tanh(self.goal_encoder(goal))

        phi_s = F.normalize(phi_s, p=2, dim=-1, eps=1e-8)
        psi_z = F.normalize(psi_z, p=2, dim=-1, eps=1e-8)
        action_emb = F.normalize(torch.tanh(self.action_emb.weight), p=2, dim=-1, eps=1e-8)
        phi_sa = F.normalize(phi_s.unsqueeze(1) * action_emb.unsqueeze(0), p=2, dim=-1, eps=1e-8)
        q_vals = (phi_sa * psi_z.unsqueeze(1)).sum(dim=-1)
        
        return q_vals

In [ ]:
def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=500):
    base = FourRoomsGridWorld(room_size=5, max_episode_steps=max_horizon)
    env = FourRoomsGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
    )
    return env


In [ ]:
BUFFER_CAPACITY = 100000
GOAL = (9, 9)
LR = float(1e-3)

env = make_env(goal=GOAL)
obs_dim = env.observation_space.shape[0]
num_actions = env.action_space.n

# Factorised Q-network instead of plain DQN_QNetwork
# Goal is 2-D (grid coordinates), so goal_dim=2
q_net = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)

q_target = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)

q_target.load_state_dict(q_net.state_dict())
for p in q_target.parameters():
    p.requires_grad_(False)


def dqn_train(
    q_network=q_net,
    q_target_network=q_target,
    env=env,
    buffer_capacity=BUFFER_CAPACITY,
    lr=LR,
    obs_dim=obs_dim,
    device=DEVICE,
    total_steps=100000,
    warmup_steps=5000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=50000,
    train_freq=4,
    goal=GOAL,
    params=None,
):

    if params is None:
        opt = optim.Adam([
            {"params": q_network.obs_encoder.parameters(), "lr": 1e-3},
            {"params": q_network.action_emb.parameters(), "lr": 1e-3},
            {"params": q_network.goal_encoder.parameters(), "lr": 3e-3},
        ])
    else:
        opt = optim.Adam(params, lr=3e-3)

    buffer = TrajectoryReplayBufferDiscrete(buffer_capacity, obs_dim, 1, device=device)

    # goal tensor: broadcasted later for batches
    goal_arr = np.array(goal, dtype=np.float32)
    goal_t_single = torch.tensor(goal_arr, dtype=torch.float32, device=device).unsqueeze(0)

    obs, _ = env.reset()
    global_step = 0
    eval_returns = []

    ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = (
        [],
        [],
        [],
        [],
        [],
        [],
    )

    while global_step < total_steps:
        # Epsilon-greedy action selection
        frac = min(1.0, global_step / eps_decay_steps)
        eps = eps_start + frac * (eps_end - eps_start)

        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                q_vals = q_network(obs_t, goal_t_single)  # [1, A]
                action = int(q_vals.argmax(dim=-1).item())

        next_obs, rew, term, trunc, _ = env.step(action)
        done = term or trunc

        ep_obs.append(obs.copy())
        ep_actions.append(action)
        ep_rewards.append(float(rew))
        ep_next_obs.append(next_obs.copy())
        ep_terminated.append(float(term))
        ep_truncated.append(float(trunc))

        obs = next_obs
        global_step += 1

        if done:
            episode = {
                "obs": ep_obs,
                "actions": ep_actions,
                "rewards": ep_rewards,
                "next_obs": ep_next_obs,
                "terminated": ep_terminated,
                "truncated": ep_truncated,
            }
            buffer.add_episode(episode)
            ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = (
                [],
                [],
                [],
                [],
                [],
                [],
            )
            obs, _ = env.reset()

        # Training update
        if len(buffer) >= warmup_steps and global_step % train_freq == 0:
            batch = buffer.sample(batch_size)

            obs_t = batch.obs                             # [B, obs_dim]
            act_t = batch.actions.long()                  # [B, 1]
            rew_t = batch.rewards                         # [B, 1]
            next_obs_t = batch.next_obs                   # [B, obs_dim]
            term_t = batch.terminated                     # [B, 1]

            # Build goal batch: same goal repeated for each transition
            goal_batch = goal_t_single.expand(obs_t.shape[0], -1)  # [B, goal_dim]

            with torch.no_grad():
                # future discounted return term: gamma * max_{a'} Q^-(s',a';z)
                next_q_vals = q_target_network(next_obs_t, goal_batch)  # [B, A]
                next_q = next_q_vals.max(dim=-1, keepdim=True).values   # [B, 1]
                target = rew_t + gamma * (1.0 - term_t) * next_q        # [B, 1]

                psi_norm = q_network.goal_encoder(goal_batch).norm(dim=-1).mean().item()
                if global_step % 5000 == 0:
                    print("psi norm", psi_norm)

            # current Q(s,a;z)
            current_q_all = q_network(obs_t, goal_batch)                # [B, A]
            current_q = current_q_all.gather(1, act_t.unsqueeze(1))     # [B, 1]

            loss = F.mse_loss(current_q, target)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_network.parameters(), 10.0)
            opt.step()

            # Soft update target network
            for p, p_tgt in zip(q_network.parameters(), q_target_network.parameters()):
                p_tgt.data.mul_(1.0 - tau).add_(tau * p.data)

        if global_step % 1000 == 0:
            eval_env = make_env(goal)
            goal_eval_arr = np.array(goal, dtype=np.float32)
            goal_eval_t = torch.tensor(goal_eval_arr, dtype=torch.float32, device=device).unsqueeze(0)

            def eval_policy(o):
                o_t = torch.tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    q_vals = q_network(o_t, goal_eval_t)
                    return int(q_vals.argmax(dim=-1).item())

            mean_ret, mean_len = evaluate_policy(eval_env, eval_policy, episodes=8)
            eval_returns.append((global_step, mean_ret))
            # print(
            #     f"[DQN-factorised] step={global_step:7d} | eps={eps:.3f} "
            #     f"| eval_return={mean_ret:.3f} | eval_len={mean_len:.1f}"
            # )
            eval_env.close()

    env.close()
    return q_network, eval_returns


# Train on the original goal
dqn_q_first, dqn_eval_first = dqn_train()

if dqn_eval_first:
    xs, ys = zip(*dqn_eval_first)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title("Factorised DQN on FourRooms (discrete)")
    plt.grid(alpha=0.25)
    plt.show()

## Visualisations

In [ ]:
dqn_q_first.eval()
eval_env_first = make_env(goal=GOAL)


def dqn_policy_fn(obs, q_network, goal):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    goal_arr = np.array(goal, dtype=np.float32)
    goal_t = torch.tensor(goal_arr, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        q_vals = q_network(obs_t, goal_t)
        action = int(q_vals.argmax(dim=-1).item())
    return action


def dqn_value_fn(obs_batch, q_network, goal):
    obs_t = torch.tensor(obs_batch, dtype=torch.float32, device=DEVICE)
    B = obs_t.shape[0]
    goal_arr = np.array(goal, dtype=np.float32)
    goal_t = torch.tensor(goal_arr, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    goal_batch = goal_t.expand(B, -1)
    with torch.no_grad():
        q_vals = q_network(obs_t.to(DEVICE), goal_batch.to(DEVICE)).cpu().numpy()
    return q_vals


plot_policy_rollouts(
    env=eval_env_first,
    policy_fn=lambda obs: dqn_policy_fn(obs, q_network=dqn_q_first, goal=GOAL),
    goal_pos=GOAL,
    eval_episodes=8,
    n_cols=4,
    is_discrete=True,
    step_point_size=12,
    start_size=60,
    end_size=50,
    goal_size=130,
    arrow_width=0.01,
)

plot_q_diagnostics(
    env=eval_env_first,
    value_fn=lambda obs_batch: dqn_value_fn(obs_batch, q_network=dqn_q_first, goal=GOAL),
    actor_fn=None,
    is_discrete=True,
    num_actions=4,
    action_names=["Up", "Down", "Left", "Right"],
    goal_pos=GOAL,
    eval_returns=dqn_eval_first,
)

eval_env_first.close()
dqn_q_first.train()

## Now to check how much samples needed to reach a new goal

In [ ]:
NEW_GOAL = (9, 1)

new_env = make_env(goal=NEW_GOAL)

# New target network initialised from the first trained factorised network
q_target_new = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)
q_target_new.load_state_dict(dqn_q_first.state_dict())
for p in q_target_new.parameters():
    p.requires_grad_(False)

# Freeze environment representation (phi) and action embedding in q_net;
# only allow psi (goal encoder) to adapt to the new goal.
# for p in dqn_q_first.obs_encoder.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_first.action_emb.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_first.goal_encoder.parameters():
#     p.requires_grad_(True)

# Train only psi(goal) parameters on the new goal
dqn_q_new, dqn_eval_new = dqn_train(
    env=new_env,
    q_network=dqn_q_first,             # same network, re-used env representation
    q_target_network=q_target_new,
    goal=NEW_GOAL,
    # params=dqn_q_first.goal_encoder.parameters(),  # only psi updated
)

if dqn_eval_new:
    xs, ys = zip(*dqn_eval_new)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title("Factorised DQN on FourRooms (discrete) on top of pre-trained phi")
    plt.grid(alpha=0.25)
    plt.show()

In [ ]:

dqn_q_new.eval()
eval_env_new = make_env(goal=NEW_GOAL)


plot_policy_rollouts(
    env=eval_env_new,
    policy_fn=lambda obs: dqn_policy_fn(obs, q_network=dqn_q_new, goal=NEW_GOAL),
    goal_pos=NEW_GOAL,
    eval_episodes=8,
    n_cols=4,
    is_discrete=True,
    step_point_size=12,
    start_size=60,
    end_size=50,
    goal_size=130,
    arrow_width=0.01,
)

plot_q_diagnostics(
    env=eval_env_new,
    value_fn=lambda obs_batch: dqn_value_fn(obs_batch, q_network=dqn_q_new, goal=NEW_GOAL),
    actor_fn=None,
    is_discrete=True,
    num_actions=4,
    action_names=['Up', 'Down', 'Left', 'Right'],
    goal_pos=NEW_GOAL,
    eval_returns=dqn_eval_new,
)

eval_env_new.close()


In [ ]:
NEW_GOAL = (1, 9)

new_env = make_env(goal=NEW_GOAL)

# New target network initialised from the first trained factorised network
q_target_new = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)
q_target_new.load_state_dict(dqn_q_new.state_dict())
for p in q_target_new.parameters():
    p.requires_grad_(False)

# Freeze environment representation (phi) and action embedding in q_net;
# only allow psi (goal encoder) to adapt to the new goal.
# for p in dqn_q_new.obs_encoder.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_new.action_emb.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_new.goal_encoder.parameters():
#     p.requires_grad_(True)

# Train only psi(goal) parameters on the new goal
dqn_q_new, dqn_eval_new = dqn_train(
    env=new_env,
    q_network=dqn_q_new,             # same network, re-used env representation
    q_target_network=q_target_new,
    goal=NEW_GOAL,
    # params=dqn_q_new.goal_encoder.parameters(),  # only psi updated
)

if dqn_eval_new:
    xs, ys = zip(*dqn_eval_new)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title("Factorised DQN on FourRooms (discrete) on top of pre-trained phi")
    plt.grid(alpha=0.25)
    plt.show()


dqn_q_new.eval()
eval_env_new = make_env(goal=NEW_GOAL)


plot_policy_rollouts(
    env=eval_env_new,
    policy_fn=lambda obs: dqn_policy_fn(obs, q_network=dqn_q_new, goal=NEW_GOAL),
    goal_pos=NEW_GOAL,
    eval_episodes=8,
    n_cols=4,
    is_discrete=True,
    step_point_size=12,
    start_size=60,
    end_size=50,
    goal_size=130,
    arrow_width=0.01,
)

plot_q_diagnostics(
    env=eval_env_new,
    value_fn=lambda obs_batch: dqn_value_fn(obs_batch, q_network=dqn_q_new, goal=NEW_GOAL),
    actor_fn=None,
    is_discrete=True,
    num_actions=4,
    action_names=['Up', 'Down', 'Left', 'Right'],
    goal_pos=NEW_GOAL,
    eval_returns=dqn_eval_new,
)

eval_env_new.close()


In [ ]:
NEW_GOAL = (1, 1)

new_env = make_env(goal=NEW_GOAL)

# New target network initialised from the first trained factorised network
q_target_new = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)
q_target_new.load_state_dict(dqn_q_new.state_dict())
for p in q_target_new.parameters():
    p.requires_grad_(False)

# Freeze environment representation (phi) and action embedding in q_net;
# only allow psi (goal encoder) to adapt to the new goal.
# for p in dqn_q_new.obs_encoder.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_new.action_emb.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_new.goal_encoder.parameters():
#     p.requires_grad_(True)

# Train only psi(goal) parameters on the new goal
dqn_q_new, dqn_eval_new = dqn_train(
    env=new_env,
    q_network=dqn_q_new,             # same network, re-used env representation
    q_target_network=q_target_new,
    goal=NEW_GOAL,
    # params=dqn_q_new.goal_encoder.parameters(),  # only psi updated
)

if dqn_eval_new:
    xs, ys = zip(*dqn_eval_new)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title("Factorised DQN on FourRooms (discrete) on top of pre-trained phi")
    plt.grid(alpha=0.25)
    plt.show()


dqn_q_new.eval()
eval_env_new = make_env(goal=NEW_GOAL)


plot_policy_rollouts(
    env=eval_env_new,
    policy_fn=lambda obs: dqn_policy_fn(obs, q_network=dqn_q_new, goal=NEW_GOAL),
    goal_pos=NEW_GOAL,
    eval_episodes=8,
    n_cols=4,
    is_discrete=True,
    step_point_size=12,
    start_size=60,
    end_size=50,
    goal_size=130,
    arrow_width=0.01,
)

plot_q_diagnostics(
    env=eval_env_new,
    value_fn=lambda obs_batch: dqn_value_fn(obs_batch, q_network=dqn_q_new, goal=NEW_GOAL),
    actor_fn=None,
    is_discrete=True,
    num_actions=4,
    action_names=['Up', 'Down', 'Left', 'Right'],
    goal_pos=NEW_GOAL,
    eval_returns=dqn_eval_new,
)

eval_env_new.close()


In [ ]:
NEW_GOAL = (9, 8)

new_env = make_env(goal=NEW_GOAL)

# New target network initialised from the first trained factorised network
q_target_new = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)
q_target_new.load_state_dict(dqn_q_new.state_dict())
for p in q_target_new.parameters():
    p.requires_grad_(False)

# Freeze environment representation (phi) and action embedding in q_net;
# only allow psi (goal encoder) to adapt to the new goal.
# for p in dqn_q_new.obs_encoder.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_new.action_emb.parameters():
#     p.requires_grad_(False)
# for p in dqn_q_new.goal_encoder.parameters():
#     p.requires_grad_(True)

# Train only psi(goal) parameters on the new goal
dqn_q_new, dqn_eval_new = dqn_train(
    env=new_env,
    q_network=dqn_q_new,             # same network, re-used env representation
    q_target_network=q_target_new,
    goal=NEW_GOAL,
    # params=dqn_q_new.goal_encoder.parameters(),  # only psi updated
)

if dqn_eval_new:
    xs, ys = zip(*dqn_eval_new)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title("Factorised DQN on FourRooms (discrete) on top of pre-trained phi")
    plt.grid(alpha=0.25)
    plt.show()


dqn_q_new.eval()
eval_env_new = make_env(goal=NEW_GOAL)


plot_policy_rollouts(
    env=eval_env_new,
    policy_fn=lambda obs: dqn_policy_fn(obs, q_network=dqn_q_new, goal=NEW_GOAL),
    goal_pos=NEW_GOAL,
    eval_episodes=8,
    n_cols=4,
    is_discrete=True,
    step_point_size=12,
    start_size=60,
    end_size=50,
    goal_size=130,
    arrow_width=0.01,
)

plot_q_diagnostics(
    env=eval_env_new,
    value_fn=lambda obs_batch: dqn_value_fn(obs_batch, q_network=dqn_q_new, goal=NEW_GOAL),
    actor_fn=None,
    is_discrete=True,
    num_actions=4,
    action_names=['Up', 'Down', 'Left', 'Right'],
    goal_pos=NEW_GOAL,
    eval_returns=dqn_eval_new,
)

eval_env_new.close()
